In [1]:
import numpy as np
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Load Model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.2996, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [25]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

device(type='cuda', index=1)

In [26]:
ref_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
ref_model.set_use_module_grafting(False)
ref_model.set_use_scc_status(False)

In [28]:
outputs = ref_model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.6582, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [29]:
ref_model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

### Load Trained Weights

In [30]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [31]:
from train_eval_func import eval_loop

In [32]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [33]:
ref_model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [34]:
eval_loop(ref_model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [29]:
model.bert.encoder.layer[0].attention.self.query

Linear(in_features=768, out_features=768, bias=True)

In [30]:
model.bert.encoder.layer[0].attention.self.query.bias.shape

torch.Size([768])

In [31]:
model.bert.encoder.layer[0].attention.output

BertSelfOutput(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [32]:
model.bert.encoder.layer[0].attention.output.dense.bias.shape

torch.Size([768])

In [33]:
model.bert.encoder.layer[0].intermediate

BertIntermediate(
  (dense): Linear(in_features=768, out_features=3072, bias=True)
  (intermediate_act_fn): GELUActivation()
)

In [34]:
model.bert.encoder.layer[0].intermediate.dense.bias.shape

torch.Size([3072])

# 5. Helper Functions

In [35]:
def reorder_merged_layer_heads(merged_layer, head_importance, device):
    """
    Reorder heads in merged layer so most important are first.
    """
    n_heads = merged_layer.attention.self.num_attention_heads
    head_size = merged_layer.attention.self.attention_head_size
    
    # Ensure head_importance is on the correct device
    head_importance = head_importance.to(device)
    
    # Get sorted indices (descending importance)
    sorted_head_indices = torch.argsort(head_importance, descending=True)
    
    # Create reordering index for weights [all_head_size]
    # Create directly on device to avoid transfer
    index = torch.arange(n_heads * head_size, device=device).view(n_heads, head_size)
    index = index[sorted_head_indices].view(-1).contiguous()
    
    def reorder_layer(linear_layer, index, dim):
        W = linear_layer.weight.index_select(dim, index).clone()
        
        if linear_layer.bias is not None:
            if dim == 1:
                b = linear_layer.bias.clone()
            else:
                b = linear_layer.bias[index].clone()
        
        linear_layer.weight.data = W.contiguous()
        if linear_layer.bias is not None:
            linear_layer.bias.data = b.contiguous()
    
    # Reorder Q, K, V (output dimension)
    reorder_layer(merged_layer.attention.self.query, index, dim=0)
    reorder_layer(merged_layer.attention.self.key, index, dim=0)
    reorder_layer(merged_layer.attention.self.value, index, dim=0)
    
    # Reorder output (input dimension)
    reorder_layer(merged_layer.attention.output.dense, index, dim=1)
    
    return merged_layer

In [36]:
def reorder_merged_layer_neurons(merged_layer, neuron_importance, device):
    """
    Reorder FFN neurons so most important are first.
    """
    # Ensure neuron_importance is on the correct device
    neuron_importance = neuron_importance.to(device)
    
    # Get sorted indices (descending importance)
    sorted_neuron_indices = torch.argsort(neuron_importance, descending=True)
    
    def reorder_layer(linear_layer, index, dim):
        W = linear_layer.weight.index_select(dim, index).clone()
        
        if linear_layer.bias is not None:
            if dim == 1:
                b = linear_layer.bias.clone()
            else:
                b = linear_layer.bias[index].clone()
        
        linear_layer.weight.data = W.contiguous()
        if linear_layer.bias is not None:
            linear_layer.bias.data = b.contiguous()
    
    # Reorder intermediate (output dimension)
    reorder_layer(merged_layer.intermediate.dense, sorted_neuron_indices, dim=0)
    
    # Reorder output (input dimension)
    reorder_layer(merged_layer.output.dense, sorted_neuron_indices, dim=1)
    
    return merged_layer

In [37]:
from structure_prune_helper import compute_importance_scores, register_importance_masks, remove_importance_masks

In [38]:
register_importance_masks(model, device, register_heads=True, register_ffn=True)

  Layer 0: Registered masks (12 heads, 3072 neurons)
  Layer 1: Registered masks (12 heads, 3072 neurons)
  Layer 2: Registered masks (12 heads, 3072 neurons)
  Layer 3: Registered masks (12 heads, 3072 neurons)
  Layer 4: Registered masks (12 heads, 3072 neurons)
  Layer 5: Registered masks (12 heads, 3072 neurons)
  Layer 6: Registered masks (12 heads, 3072 neurons)
  Layer 7: Registered masks (12 heads, 3072 neurons)
  Layer 8: Registered masks (12 heads, 3072 neurons)
  Layer 9: Registered masks (12 heads, 3072 neurons)
  Layer 10: Registered masks (12 heads, 3072 neurons)
  Layer 11: Registered masks (12 heads, 3072 neurons)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [39]:
head_importance, neuron_importance = compute_importance_scores(model, train_dataloader, device, compute_heads=True, compute_ffn=True)

Computing importance (heads + FFN): 100%|██████████| 115/115 [00:12<00:00,  9.18it/s]


In [40]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [41]:
for idx, layer in enumerate(model.bert.encoder.layer):
    reorder_merged_layer_heads(layer, head_importance[idx], device)
    reorder_merged_layer_neurons(layer, neuron_importance[idx], device)

In [42]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [43]:
for idx, layer in enumerate(ref_model.bert.encoder.layer):
    reorder_merged_layer_heads(layer, head_importance[idx], device)
    reorder_merged_layer_neurons(layer, neuron_importance[idx], device)

In [44]:
eval_loop(ref_model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [45]:
import torch.nn as nn

In [46]:
def resize_qkv(linear_layer, new_out_dim):
    new_layer = nn.Linear(
        in_features=linear_layer.in_features,
        out_features=new_out_dim,
        bias=True
    ).to(linear_layer.weight.device)

    return new_layer


def resize_output(linear_layer, new_in_dim):
    new_layer = nn.Linear(
        in_features=new_in_dim,
        out_features=linear_layer.out_features,
        bias=True
    ).to(linear_layer.weight.device)

    return new_layer

In [47]:
def cw_helper(sourceLinearLayer, targetLinearLayer, s_len, t_len, dim=0):
    with torch.no_grad():

        if dim == 0:
            # Row-wise (Q, K, V)
            W_source = sourceLinearLayer.weight.data[:s_len]
            W_target = targetLinearLayer.weight.data[:t_len]

            b_source = sourceLinearLayer.bias.data[:s_len]
            b_target = targetLinearLayer.bias.data[:t_len]

            new_W = torch.cat([W_target, W_source], dim=0)
            new_b = torch.cat([b_target, b_source], dim=0)

            targetLinearLayer.weight.data[:t_len + s_len] = new_W
            targetLinearLayer.bias.data[:t_len + s_len] = new_b

        else:
            # Column-wise (output projection)
            W_source = sourceLinearLayer.weight.data[:, :s_len]
            W_target = targetLinearLayer.weight.data[:, :t_len]

            new_W = torch.cat([W_target, W_source], dim=1)

            targetLinearLayer.weight.data[:, :t_len + s_len] = new_W

            # Bias unchanged (correct for output layer)

In [48]:
def merge_mha(model, sourceLayer, targetLayer,
              source_head_imp_sorted, target_head_imp_sorted,
              device, extra_heads=0):

    s_n_head = sourceLayer.attention.self.num_attention_heads
    t_n_head = targetLayer.attention.self.num_attention_heads
    head_size = sourceLayer.attention.self.attention_head_size

    # Final number of heads
    n_head = max(s_n_head, t_n_head) + extra_heads
    new_dim = n_head * head_size

    # ---------------------------
    # Decide how many heads to take
    # ---------------------------
    s_idx, t_idx = 0, 0

    for _ in range(n_head):

        if s_idx >= s_n_head:
            t_idx += 1
            continue

        if t_idx >= t_n_head:
            s_idx += 1
            continue

        if source_head_imp_sorted[s_idx] > target_head_imp_sorted[t_idx]:
            s_idx += 1
        else:
            t_idx += 1

    print(s_idx, t_idx)

    # Convert head count → dimension
    s_len = s_idx * head_size
    t_len = new_dim - s_len

    # ---------------------------
    # Resize layers FIRST
    # ---------------------------
    targetLayer.attention.self.query = resize_qkv(
        targetLayer.attention.self.query, new_dim
    )
    targetLayer.attention.self.key = resize_qkv(
        targetLayer.attention.self.key, new_dim
    )
    targetLayer.attention.self.value = resize_qkv(
        targetLayer.attention.self.value, new_dim
    )

    targetLayer.attention.output.dense = resize_output(
        targetLayer.attention.output.dense, new_dim
    )

    # ---------------------------
    # Merge weights
    # ---------------------------
    cw_helper(sourceLayer.attention.self.query,
              targetLayer.attention.self.query,
              s_len, t_len)

    cw_helper(sourceLayer.attention.self.key,
              targetLayer.attention.self.key,
              s_len, t_len)

    cw_helper(sourceLayer.attention.self.value,
              targetLayer.attention.self.value,
              s_len, t_len)

    cw_helper(sourceLayer.attention.output.dense,
              targetLayer.attention.output.dense,
              s_len, t_len,
              dim=1)

    # ---------------------------
    # Update config
    # ---------------------------
    targetLayer.attention.self.num_attention_heads = n_head
    targetLayer.attention.self.all_head_size = new_dim

    # ---------------------------
    # Update head mask
    # ---------------------------
    targetLayer.head_mask_param = nn.Parameter(
        torch.ones(n_head, dtype=torch.float32, device=device),
        requires_grad=True
    )              

In [49]:
def cw_ff_helper(sourceLinearLayer, targetLinearLayer, s_len, t_len, dim=0):
    with torch.no_grad():

        if dim == 0:
            # Row-wise (intermediate layer)
            W_source = sourceLinearLayer.weight.data[:s_len]
            W_target = targetLinearLayer.weight.data[:t_len]

            b_source = sourceLinearLayer.bias.data[:s_len]
            b_target = targetLinearLayer.bias.data[:t_len]

            new_W = torch.cat([W_target, W_source], dim=0)
            new_b = torch.cat([b_target, b_source], dim=0)

            targetLinearLayer.weight.data[:t_len + s_len] = new_W
            targetLinearLayer.bias.data[:t_len + s_len] = new_b

        else:
            # Column-wise (output layer)
            W_source = sourceLinearLayer.weight.data[:, :s_len]
            W_target = targetLinearLayer.weight.data[:, :t_len]

            new_W = torch.cat([W_target, W_source], dim=1)

            targetLinearLayer.weight.data[:, :t_len + s_len] = new_W

            # Bias unchanged

In [50]:
def resize_ffn_intermediate(linear_layer, new_out_dim):
    return nn.Linear(
        linear_layer.in_features,
        new_out_dim,
        bias=True
    ).to(linear_layer.weight.device)


def resize_ffn_output(linear_layer, new_in_dim):
    return nn.Linear(
        new_in_dim,
        linear_layer.out_features,
        bias=True
    ).to(linear_layer.weight.device)

In [83]:
def merge_ff(model, sourceLayer, targetLayer,
             source_int_imp_sorted,
             target_int_imp_sorted,
             device,
             extra_neurons=0):

    s_dim = sourceLayer.intermediate.dense.out_features
    t_dim = targetLayer.intermediate.dense.out_features

    # Final FFN size
    f_dim = max(s_dim, t_dim) + extra_neurons

    s_idx, t_idx = 0, 0
                 
    # ---------------------------
    # Decide allocation
    # ---------------------------
    for _ in range(f_dim):

        if s_idx >= s_dim:
            t_idx += 1
            continue

        if t_idx >= t_dim:
            s_idx += 1
            continue

        if source_int_imp_sorted[s_idx] > target_int_imp_sorted[t_idx]:
            s_idx += 1
        else:
            t_idx += 1

    print(s_idx, t_idx)

    s_len = s_idx
    t_len = f_dim - s_len

    # ---------------------------
    # Resize layers FIRST
    # ---------------------------
    targetLayer.intermediate.dense = resize_ffn_intermediate(
        targetLayer.intermediate.dense,
        f_dim
    )

    targetLayer.output.dense = resize_ffn_output(
        targetLayer.output.dense,
        f_dim
    )

    # ---------------------------
    # Merge weights
    # ---------------------------
    cw_ff_helper(
        sourceLayer.intermediate.dense,
        targetLayer.intermediate.dense,
        s_len, t_len
    )

    cw_ff_helper(
        sourceLayer.output.dense,
        targetLayer.output.dense,
        s_len, t_len,
        dim=1
    )

    # ---------------------------
    # Update config
    # ---------------------------
    targetLayer.intermediate.dense.out_features = f_dim

    # ---------------------------
    # Update FFN mask
    # ---------------------------
    targetLayer.int_mask_param = nn.Parameter(
        torch.ones(f_dim, dtype=torch.float32, device=device),
        requires_grad=True
    )

In [52]:
head_imp1, indices1 = torch.sort(head_importance[2], descending=True)

In [53]:
head_imp2, indices2 = torch.sort(head_importance[3], descending=True)

In [54]:
neuron_imp1, indices1 = torch.sort(neuron_importance[2], descending=True)

In [55]:
neuron_imp2, indices1 = torch.sort(neuron_importance[3], descending=True)

In [61]:
model.bert.encoder.layer[2].attention.self.query.weight == ref_model.bert.encoder.layer[2].attention.self.query.weight

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]], device='cuda:1')

In [65]:
model.bert.encoder.layer[2].intermediate.dense.weight == ref_model.bert.encoder.layer[2].intermediate.dense.weight

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]], device='cuda:1')

In [66]:
merge_mha(model, model.bert.encoder.layer[2], model.bert.encoder.layer[3], head_imp1, head_imp2, device, 4)

9 7


In [81]:
def test_mha_merge(
    model,              # merged model
    ref_model,          # clone BEFORE merge
    source_layer_idx,
    target_layer_idx,
    s_idx, t_idx,       # number of heads taken
):
    layer_m = model.bert.encoder.layer[target_layer_idx]
    layer_s = ref_model.bert.encoder.layer[source_layer_idx]
    layer_t = ref_model.bert.encoder.layer[target_layer_idx]

    head_size = layer_m.attention.self.attention_head_size

    # Convert heads → rows
    s_len = s_idx * head_size
    t_len = t_idx * head_size

    print(f"\nTesting MHA Merge:")
    print(f"Target heads kept: {t_idx}, Source heads copied: {s_idx}")

    # ---------------------------
    # 1. TEST TARGET PART
    # ---------------------------
    print("\n[1] Checking TARGET part...")

    for h in range(t_idx):
        start = h * head_size
        end   = (h + 1) * head_size

        ok = torch.allclose(
            layer_m.attention.self.query.weight[start:end],
            layer_t.attention.self.query.weight[start:end]
        )

        print(f"  Head {h} (target) match: {ok}")

    # ---------------------------
    # 2. TEST SOURCE PART
    # ---------------------------
    print("\n[2] Checking SOURCE part...")

    for h in range(s_idx):
        src_start = h * head_size
        src_end   = (h + 1) * head_size

        tgt_start = t_len + h * head_size
        tgt_end   = tgt_start + head_size

        ok = torch.allclose(
            layer_m.attention.self.query.weight[tgt_start:tgt_end],
            layer_s.attention.self.query.weight[src_start:src_end]
        )

        print(f"  Head {h} (source → merged) match: {ok}")

    # ---------------------------
    # 3. FINAL CHECK
    # ---------------------------
    print("\n[3] Summary:")
    print("✔ If all True → merge is correct")

In [82]:
test_mha_merge(
    model,
    ref_model,
    source_layer_idx=2,
    target_layer_idx=3,
    s_idx=9,
    t_idx=7
)


Testing MHA Merge:
Target heads kept: 7, Source heads copied: 9

[1] Checking TARGET part...
  Head 0 (target) match: False
  Head 1 (target) match: False
  Head 2 (target) match: False
  Head 3 (target) match: False
  Head 4 (target) match: False
  Head 5 (target) match: False
  Head 6 (target) match: False

[2] Checking SOURCE part...
  Head 0 (source → merged) match: True
  Head 1 (source → merged) match: True
  Head 2 (source → merged) match: True
  Head 3 (source → merged) match: True
  Head 4 (source → merged) match: True
  Head 5 (source → merged) match: True
  Head 6 (source → merged) match: True
  Head 7 (source → merged) match: True
  Head 8 (source → merged) match: True

[3] Summary:
✔ If all True → merge is correct


In [74]:
model.bert.encoder.layer[3].attention.self.query.weight.shape

torch.Size([1024, 768])

In [84]:
merge_ff(model, model.bert.encoder.layer[2], model.bert.encoder.layer[3], neuron_imp1, neuron_imp2, device, extra_neurons=1000)

2010 2062


In [85]:
def test_ffn_merge(
    model,              # merged model
    ref_model,          # clone BEFORE merge
    source_layer_idx,
    target_layer_idx,
    s_idx, t_idx
):
    layer_m = model.bert.encoder.layer[target_layer_idx]
    layer_s = ref_model.bert.encoder.layer[source_layer_idx]
    layer_t = ref_model.bert.encoder.layer[target_layer_idx]

    s_len = s_idx
    t_len = t_idx

    print(f"\nTesting FFN Merge:")
    print(f"Target neurons kept: {t_idx}, Source neurons copied: {s_idx}")

    # ---------------------------
    # 1. INTERMEDIATE LAYER (row-wise)
    # ---------------------------
    print("\n[1] Checking INTERMEDIATE (dense) layer...")

    # Target part
    ok_target = torch.allclose(
        layer_m.intermediate.dense.weight[:t_len],
        layer_t.intermediate.dense.weight[:t_len]
    )
    print(f"  Target part match: {ok_target}")

    # Source part
    ok_source = torch.allclose(
        layer_m.intermediate.dense.weight[t_len:t_len+s_len],
        layer_s.intermediate.dense.weight[:s_len]
    )
    print(f"  Source part match: {ok_source}")

    # ---------------------------
    # 2. OUTPUT LAYER (column-wise)
    # ---------------------------
    print("\n[2] Checking OUTPUT (dense) layer...")

    # Target part
    ok_target_out = torch.allclose(
        layer_m.output.dense.weight[:, :t_len],
        layer_t.output.dense.weight[:, :t_len]
    )
    print(f"  Target part match: {ok_target_out}")

    # Source part
    ok_source_out = torch.allclose(
        layer_m.output.dense.weight[:, t_len:t_len+s_len],
        layer_s.output.dense.weight[:, :s_len]
    )
    print(f"  Source part match: {ok_source_out}")

    # ---------------------------
    # 3. BIAS CHECK (intermediate only)
    # ---------------------------
    print("\n[3] Checking INTERMEDIATE bias...")

    ok_bias_target = torch.allclose(
        layer_m.intermediate.dense.bias[:t_len],
        layer_t.intermediate.dense.bias[:t_len]
    )
    print(f"  Target bias match: {ok_bias_target}")

    ok_bias_source = torch.allclose(
        layer_m.intermediate.dense.bias[t_len:t_len+s_len],
        layer_s.intermediate.dense.bias[:s_len]
    )
    print(f"  Source bias match: {ok_bias_source}")

    # ---------------------------
    # FINAL SUMMARY
    # ---------------------------
    print("\n[4] Summary:")
    print("✔ All True → FFN merge is correct")

In [86]:
test_ffn_merge(
    model,
    ref_model,
    source_layer_idx=2,
    target_layer_idx=3,
    s_idx=2010,
    t_idx=2062
)


Testing FFN Merge:
Target neurons kept: 2062, Source neurons copied: 2010

[1] Checking INTERMEDIATE (dense) layer...
  Target part match: False
  Source part match: True

[2] Checking OUTPUT (dense) layer...
  Target part match: False
  Source part match: True

[3] Checking INTERMEDIATE bias...
  Target bias match: False
  Source bias match: True

[4] Summary:
✔ All True → FFN merge is correct


In [57]:
model.bert.encoder.layer[2]

BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3072, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): BertOutput(
    (dense): Linear(in_features=3072, out_features=768, bias=True)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [58]:
model.bert.encoder.layer[3]

BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=1024, bias=True)
      (key): Linear(in_features=768, out_features=1024, bias=True)
      (value): Linear(in_features=768, out_features=1024, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=1024, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=4072, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): BertOutput(
    (dense): Linear(in_features=4072, out_features=768, bias=True)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [56]:
remove_importance_masks(model)

✓ Removed head and FFN masks


In [59]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8112745098039216, 'f1': 0.8768}